# Coordinate arrays
A model declares its tensor contract without allocating field data.

In [ ]:
import numpy as np
import earth2studio as e2s
from earth2studio.models.px.fcn import VARIABLES as FCN_VARIABLES

e2s.known_grids(), e2s.resolve_grid("fcn")

## FourCastNet coordinate contract

In [ ]:
class ExampleFCN:
    step = np.timedelta64(6, "h")

    def input_coords(self):
        return e2s.coord_array(
            dims=("batch", "lead_time", "variable", "lat", "lon"),
            coords={"lead_time": [np.timedelta64(0, "h")], "variable": FCN_VARIABLES},
            dynamic=("batch",), grid="fcn",
        )

    def output_coords(self, coords):
        return coords.assign_coords(lead_time=coords.lead_time + self.step)

In [ ]:
model = ExampleFCN()
inputs = model.input_coords()
inputs.dims, inputs.shape, inputs.data.nbytes, inputs.e2s.get_grid()

## Populate latitude and longitude

In [ ]:
populated = inputs.e2s.materialize_grid_coords()
{
    "lat": populated.lat[[0, -1]].values,
    "lon": populated.lon[[0, -1]].values,
    "field_bytes": populated.data.nbytes,
}

## One forecast step

In [ ]:
outputs = model.output_coords(inputs)
outputs.dims, outputs.lead_time.values, outputs.data.nbytes

## Explicit coordinates

In [ ]:
explicit = e2s.coord_array(
    dims=inputs.dims, dynamic=("batch",), grid="fcn",
    coords={
        "lead_time": [np.timedelta64(0, "h")], "variable": FCN_VARIABLES,
        "lat": np.arange(90, -90, -0.25), "lon": np.arange(0, 360, 0.25),
    },
)
tuple(explicit.coords)

## HRRR and HPX

In [ ]:
def grid_coords(grid, spatial_dims):
    return e2s.coord_array(
        dims=("batch", "variable", *spatial_dims),
        coords={"variable": ["u10m"]}, dynamic=("batch",), grid=grid,
    ).e2s.materialize_grid_coords()

hrrr = grid_coords("hrrr", ("hrrr_y", "hrrr_x"))
hpx = grid_coords("hpx6", ("hpx",))
[(a.e2s.get_grid(), tuple(a.coords), a.data.nbytes) for a in (hrrr, hpx)]